## Install Kaggle API

In [2]:
%pip install kagglehub


[notice] A new release of pip is available: 24.3.1 -> 25.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Download Kaggle dataset

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("wordsforthewise/lending-club")

print("Path to dataset files:", path)

/Users/priscillaashleywijaya/Desktop/General/NUS_Fintech_Society/capstone/src/CreditShield/capstone/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/priscillaashleywijaya/.cache/kagglehub/datasets/wordsforthewise/lending-club/versions/3


## Requirements

In [4]:
import re
import os

import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# "magic" command to make plots show up in the notebook
%matplotlib inline 

# Data analysis

In [5]:
folders = os.listdir(path)
# Skip .xslx file, if we are ever able to upload it...
# Currently can't upload due to conflicts with other versions of the dataset on Kaggle.
folders = [f for f in folders if 'xlsx' not in f]
folders

['rejected_2007_to_2018q4.csv',
 'accepted_2007_to_2018q4.csv',
 'rejected_2007_to_2018Q4.csv.gz',
 'accepted_2007_to_2018Q4.csv.gz']

In [6]:
os.listdir(path + "/" + folders[1])

['accepted_2007_to_2018Q4.csv']

In [7]:
acc_folder = path + "/" + [f for f in folders if 'accepted' in f][0]
accepted_fn = acc_folder + '/' + os.listdir(acc_folder)[0]

rej_folder = path + "/" + [f for f in folders if 'rejected' in f][0]
rejected_fn = rej_folder + '/' + os.listdir(rej_folder)[0]

accepted_fn

'/Users/priscillaashleywijaya/.cache/kagglehub/datasets/wordsforthewise/lending-club/versions/3/accepted_2007_to_2018q4.csv/accepted_2007_to_2018Q4.csv'

#### check if the actual file is still there

In [8]:
if os.path.isfile(accepted_fn) and os.path.isfile(rejected_fn):
    print('both paths still point to the actual file; all is good')
else:
    print('Kaggle changed how they handle compressed files again...you need to locate the files')

both paths still point to the actual file; all is good


In [9]:
# Takes a while to read, because these files are large...give it a minute or so
acc_df = pd.read_csv(accepted_fn)

# this is a dataset with rejected loans from lendingclub
rej_df = pd.read_csv(rejected_fn)

/var/folders/tg/g161x0m50lxgx68hfwk_3qk80000gn/T/ipykernel_23605/3491062877.py:2: DtypeWarning: Columns (0,19,49,59,118,129,130,131,134,135,136,139,145,146,147) have mixed types. Specify dtype option on import or set low_memory=False.
  acc_df = pd.read_csv(accepted_fn)


In [10]:
acc_df.shape 

(2260701, 151)

* 2260701 rows, 151 columns in accepted file

In [11]:
rej_df.shape

(27648741, 9)

* 27648741 rows, 9 columns in accepted file

### FICO Score
- FICO stands for Fair Isaac Corporation, the company that created one of the most widely used credit scoring systems in the United States.

- A FICO score is a numerical measure of a person's creditworthiness — basically, how likely they are to repay a loan.

In [12]:
# fico score in accepted loans
[col for col in acc_df.columns if 'fico' in col.lower()]

['fico_range_low',
 'fico_range_high',
 'last_fico_range_high',
 'last_fico_range_low',
 'sec_app_fico_range_low',
 'sec_app_fico_range_high']

In [13]:
# fico score in rejected loans
[col for col in rej_df.columns if 'fico' in col.lower()]

[]

### Rejected information

In [14]:
rej_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27648741 entries, 0 to 27648740
Data columns (total 9 columns):
 #   Column                Dtype  
---  ------                -----  
 0   Amount Requested      float64
 1   Application Date      object 
 2   Loan Title            object 
 3   Risk_Score            float64
 4   Debt-To-Income Ratio  object 
 5   Zip Code              object 
 6   State                 object 
 7   Employment Length     object 
 8   Policy Code           float64
dtypes: float64(3), object(6)
memory usage: 1.9+ GB


## Accepted information

In [15]:
acc_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 2.5+ GB


In [16]:
pd.options.display.max_rows

60

In [17]:
acc_df.columns

Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade',
       ...
       'hardship_payoff_balance_amount', 'hardship_last_payment_amount',
       'disbursement_method', 'debt_settlement_flag',
       'debt_settlement_flag_date', 'settlement_status', 'settlement_date',
       'settlement_amount', 'settlement_percentage', 'settlement_term'],
      dtype='object', length=151)

In [18]:
# update max number of rows
pd.options.display.max_rows = 1000

In [19]:
acc_df.head().T

,0,1,2,3,4
id,68407277,68355089,68341763,66310712,68476807
member_id,NaN,NaN,NaN,NaN,NaN
loan_amnt,3600.0,24700.0,20000.0,35000.0,10400.0
funded_amnt,3600.0,24700.0,20000.0,35000.0,10400.0
funded_amnt_inv,3600.0,24700.0,20000.0,35000.0,10400.0
term,36 months,36 months,60 months,60 months,60 months
int_rate,13.99,11.99,10.78,14.85,22.45
installment,123.03,820.28,432.66,829.9,289.91
grade,C,C,B,C,F
sub_grade,C4,C1,B4,C5,F1


In [39]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
np.set_printoptions(threshold=np.inf)
acc_df['title'].unique()

array(['Debt consolidation', 'Business', nan, 'Major purchase',
       'Credit card refinancing', 'Home improvement', 'Other',
       'Home buying', 'Vacation', 'Car financing', 'Medical expenses',
       'Moving and relocation', 'Green loan', 'odymeds', 'SAVE',
       'Learning and training', 'new day',
       'Trying to come back to reality!', 'considerate',
       'Paying off higher interest cards & auto',
       'Simple Loan Until Contract Is Completed',
       'Prescription Drug and Medical Costs', 'Pay off Lowes Card',
       'new kitchen for momma!', 'DebtC',
       'New Baby and New House (CC Consolidate)',
       'Credit Card/Auto Repair', 'Student Loan',
       'Credit Card Consolidation', 'debt payoff', 'Consolidation',
       'debt pay off', 'Engagement Ring Purchase',
       'Dodger Blue not Chase blue', 'Finish line', 'Credit cards',
       'Debt Clean up', 'Getting out of debt', 'mlue',
       'The Road to Freedom', 'Debt Consolidation', 'credit cards',
       'debt', 'P

In [21]:
# .tail() shows the last few rows
acc_df.tail().T

,2260696,2260697,2260698,2260699,2260700
id,88985880,88224441,88215728,Total amount funded in policy code 1: 1465324575,Total amount funded in policy code 2: 521953170
member_id,NaN,NaN,NaN,NaN,NaN
loan_amnt,40000.0,24000.0,14000.0,NaN,NaN
funded_amnt,40000.0,24000.0,14000.0,NaN,NaN
funded_amnt_inv,40000.0,24000.0,14000.0,NaN,NaN
term,60 months,60 months,60 months,NaN,NaN
int_rate,10.49,14.49,14.49,NaN,NaN
installment,859.56,564.56,329.33,NaN,NaN
grade,B,C,C,NaN,NaN
sub_grade,B3,C4,C4,NaN,NaN


In [22]:
# .info() tells us the datatype(int64, `object` is a string)
# and will also tell us the number of non-null (not missing) data points for each column
# because this dataframe is so large, we have to force it to show the datatypes and non-null numbers with the arguments
acc_df.info(verbose = True, max_cols=None)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Data columns (total 151 columns):
 #    Column                                      Dtype  
---   ------                                      -----  
 0    id                                          object 
 1    member_id                                   float64
 2    loan_amnt                                   float64
 3    funded_amnt                                 float64
 4    funded_amnt_inv                             float64
 5    term                                        object 
 6    int_rate                                    float64
 7    installment                                 float64
 8    grade                                       object 
 9    sub_grade                                   object 
 10   emp_title                                   object 
 11   emp_length                                  object 
 12   home_ownership                              object 
 13   annual_inc

In [23]:
# shows some common summary statistics
# again, transposing with .T to make it easier to read
acc_df.describe().T

,count,mean,std,min,25%,50%,75%,max
member_id,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
loan_amnt,2260668.0,15046.931228,9190.245488,5.000000e+02,8000.0000,12900.000,20000.000000,4.000000e+04
funded_amnt,2260668.0,15041.664057,9188.413022,5.000000e+02,8000.0000,12875.000,20000.000000,4.000000e+04
funded_amnt_inv,2260668.0,15023.437745,9192.331679,0.000000e+00,8000.0000,12800.000,20000.000000,4.000000e+04
int_rate,2260668.0,13.092829,4.832138,5.310000e+00,9.4900,12.620,15.990000,3.099000e+01
installment,2260668.0,445.806823,267.173535,4.930000e+00,251.6500,377.990,593.320000,1.719830e+03
annual_inc,2260664.0,77992.428687,112696.199574,0.000000e+00,46000.0000,65000.000,93000.000000,1.100000e+08
dti,2258957.0,18.824196,14.183329,-1.000000e+00,11.8900,17.840,24.490000,9.990000e+02
delinq_2yrs,2260639.0,0.306879,0.867230,0.000000e+00,0.0000,0.000,0.000000,5.800000e+01
fico_range_low,2260668.0,698.588205,33.010376,6.100000e+02,675.0000,690.000,715.000000,8.450000e+02


## Data Cleaning

In [24]:
# 1. Get the list of column names
column_list = acc_df.columns.tolist()

# 2. Iterate and print each column name on a new line
for col in column_list:
    print(col)

id
member_id
loan_amnt
funded_amnt
funded_amnt_inv
term
int_rate
installment
grade
sub_grade
emp_title
emp_length
home_ownership
annual_inc
verification_status
issue_d
loan_status
pymnt_plan
url
desc
purpose
title
zip_code
addr_state
dti
delinq_2yrs
earliest_cr_line
fico_range_low
fico_range_high
inq_last_6mths
mths_since_last_delinq
mths_since_last_record
open_acc
pub_rec
revol_bal
revol_util
total_acc
initial_list_status
out_prncp
out_prncp_inv
total_pymnt
total_pymnt_inv
total_rec_prncp
total_rec_int
total_rec_late_fee
recoveries
collection_recovery_fee
last_pymnt_d
last_pymnt_amnt
next_pymnt_d
last_credit_pull_d
last_fico_range_high
last_fico_range_low
collections_12_mths_ex_med
mths_since_last_major_derog
policy_code
application_type
annual_inc_joint
dti_joint
verification_status_joint
acc_now_delinq
tot_coll_amt
tot_cur_bal
open_acc_6m
open_act_il
open_il_12m
open_il_24m
mths_since_rcnt_il
total_bal_il
il_util
open_rv_12m
open_rv_24m
max_bal_bc
all_util
total_rev_hi_lim
inq_fi
to

In [25]:
data = acc_df.sample(100_000, random_state=42)

In [26]:
data["loan_status"].value_counts(dropna = False)

loan_status
Fully Paid                                             47460
Current                                                38887
Charged Off                                            12034
Late (31-120 days)                                       969
In Grace Period                                          368
Late (16-30 days)                                        187
Does not meet the credit policy. Status:Fully Paid        60
Does not meet the credit policy. Status:Charged Off       31
NaN                                                        4
Name: count, dtype: int64

In [27]:
data = data.loc[data['loan_status'].isin(['Fully Paid', 'Charged Off'])].copy()
data["loan_status"].value_counts(dropna = False)

loan_status
Fully Paid     47460
Charged Off    12034
Name: count, dtype: int64

In [28]:
data.shape

(59494, 151)

In [29]:
data['loan_status'].value_counts(normalize=True, dropna=False)

loan_status
Fully Paid     0.797728
Charged Off    0.202272
Name: proportion, dtype: float64

### Retain only columns that do not leak & are known at point of application

In [30]:
print(sorted(data.columns))

['acc_now_delinq', 'acc_open_past_24mths', 'addr_state', 'all_util', 'annual_inc', 'annual_inc_joint', 'application_type', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'collection_recovery_fee', 'collections_12_mths_ex_med', 'debt_settlement_flag', 'debt_settlement_flag_date', 'deferral_term', 'delinq_2yrs', 'delinq_amnt', 'desc', 'disbursement_method', 'dti', 'dti_joint', 'earliest_cr_line', 'emp_length', 'emp_title', 'fico_range_high', 'fico_range_low', 'funded_amnt', 'funded_amnt_inv', 'grade', 'hardship_amount', 'hardship_dpd', 'hardship_end_date', 'hardship_flag', 'hardship_last_payment_amount', 'hardship_length', 'hardship_loan_status', 'hardship_payoff_balance_amount', 'hardship_reason', 'hardship_start_date', 'hardship_status', 'hardship_type', 'home_ownership', 'id', 'il_util', 'initial_list_status', 'inq_fi', 'inq_last_12m', 'inq_last_6mths', 'installment', 'int_rate', 'issue_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', '

Using information online and the best knowledge we have to retain columns that do not leak information into the future & are not irrelevant
- Things Lending Club knows at application / approval:
    - borrower info: income, employment, purpose, state
    - credit bureau info: FICO, delinquencies, utilisation, counts
    - loan terms: amount, term, interest rate, grade
    - origination date: issue_d

In [31]:
cols_to_keep = [
    # Loan terms (known before issuing the loan)
    "loan_amnt",
    "term",
    "int_rate",
    "installment",
    "funded_amnt",
    "funded_amnt_inv",
    "grade",
    "sub_grade",
    "initial_list_status",
    "disbursement_method",
    
    # Borrower attributes (application)
    "emp_title",
    "emp_length",
    "home_ownership",
    "annual_inc",
    "verification_status",
    "addr_state",
    "zip_code",
    "purpose",
    "application_type",    
    "issue_d",
    
    # Credit bureau attributes at time of application
    "dti",
    "delinq_2yrs",
    "earliest_cr_line",
    "inq_last_6mths",
    "open_acc",
    "pub_rec",
    "revol_bal",
    "revol_util",
    "total_acc",
    "fico_range_low",
    "fico_range_high",
    'pymnt_plan',

    
    # Richer bureau attributes (still at application)
    "acc_open_past_24mths",
    "bc_open_to_buy",
    "bc_util",
    "chargeoff_within_12_mths",   # applicant’s past credit behaviour
    "collections_12_mths_ex_med",
    "delinq_amnt",
    "inq_fi",
    "inq_last_12m",
    "max_bal_bc",
    "mo_sin_old_il_acct",
    "mo_sin_old_rev_tl_op",
    "mo_sin_rcnt_rev_tl_op",
    "mo_sin_rcnt_tl",
    "mort_acc",
    "mths_since_last_delinq",
    "mths_since_last_major_derog",
    "mths_since_last_record",
    "mths_since_rcnt_il",
    "num_accts_ever_120_pd",
    "num_actv_bc_tl",
    "num_actv_rev_tl",
    "num_bc_sats",
    "num_bc_tl",
    "num_il_tl",
    "num_op_rev_tl",
    "num_rev_accts",
    "num_rev_tl_bal_gt_0",
    "num_sats",
    "num_tl_120dpd_2m",
    "num_tl_30dpd",
    "num_tl_90g_dpd_24m",
    "num_tl_op_past_12m",
    "open_acc_6m",
    "open_act_il",
    "open_il_12m",
    "open_il_24m",
    "open_rv_12m",
    "open_rv_24m",
    "pct_tl_nvr_dlq",
    "percent_bc_gt_75",
    "pub_rec_bankruptcies",
    "tax_liens",
    "tot_hi_cred_lim",
    "total_bal_ex_mort",
    "total_bal_il",
    "total_bc_limit",
    "total_cu_tl",
    "total_il_high_credit_limit",
]


In [32]:
cols_to_drop = [col for col in data.columns if col not in cols_to_keep]
print(cols_to_drop)
print(len(cols_to_drop))

['id', 'member_id', 'loan_status', 'url', 'desc', 'title', 'out_prncp', 'out_prncp_inv', 'total_pymnt', 'total_pymnt_inv', 'total_rec_prncp', 'total_rec_int', 'total_rec_late_fee', 'recoveries', 'collection_recovery_fee', 'last_pymnt_d', 'last_pymnt_amnt', 'next_pymnt_d', 'last_credit_pull_d', 'last_fico_range_high', 'last_fico_range_low', 'policy_code', 'annual_inc_joint', 'dti_joint', 'verification_status_joint', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'il_util', 'all_util', 'total_rev_hi_lim', 'avg_cur_bal', 'mths_since_recent_bc', 'mths_since_recent_bc_dlq', 'mths_since_recent_inq', 'mths_since_recent_revol_delinq', 'revol_bal_joint', 'sec_app_fico_range_low', 'sec_app_fico_range_high', 'sec_app_earliest_cr_line', 'sec_app_inq_last_6mths', 'sec_app_mort_acc', 'sec_app_open_acc', 'sec_app_revol_util', 'sec_app_open_act_il', 'sec_app_num_rev_accts', 'sec_app_chargeoff_within_12_mths', 'sec_app_collections_12_mths_ex_med', 'sec_app_mths_since_last_major_derog', 'hardship_flag

In [33]:
data.drop(labels = cols_to_drop, inplace = True, axis = 1)

In [34]:
data.shape

(59494, 80)

## Pre-processing

In [35]:
# Want to determine if columns are categorical 
def analyze_categorical_features(df, unique_value_threshold=50):
    """
    Identifies and counts categories for columns that are either
    of 'object' type or have fewer unique values than the specified threshold.

    Args:
        df (pd.DataFrame): The DataFrame to analyze.
        unique_value_threshold (int): Max number of unique values a column
                                      can have to be considered categorical.

    Returns:
        dict: A dictionary where keys are column names and values are 
              their unique category counts.
    """
    categorical_counts = {}
    
    # 1. Identify candidate columns
    for col in df.columns:
        # Check if column is object type (string)
        if df[col].dtype == 'object':
            # Strings are always treated as categorical candidates
            is_categorical = True
        
        # Check if column is numeric but has a small number of unique values
        elif df[col].nunique() < unique_value_threshold:
            is_categorical = True
            
        else:
            is_categorical = False
            
        # 2. Store the count if it's a categorical candidate
        if is_categorical:
            # Drop NA values before counting unique categories
            count = df[col].nunique(dropna=True)
            categorical_counts[col] = count
            
    # Optional: Print results for easy viewing
    print(f"Found {len(categorical_counts)} categorical columns/candidates (Threshold: < {unique_value_threshold} unique values).")
    print("-" * 50)
    for col, count in sorted(categorical_counts.items(), key=lambda item: item[1], reverse=True):
        print(f"{col:<30} | Categories: {count}")
            
    return categorical_counts

# Example Usage (assuming your DataFrame is named 'data'):
# cat_summary = analyze_categorical_features(data)

In [36]:
missing_pct = acc_df.isnull().sum() / len(acc_df) * 100
high_missing_cols = missing_pct[missing_pct > 50].sort_values(ascending=False)
print(high_missing_cols)
print(f"\nTotal number of high-missing columns: {len(high_missing_cols)}")

member_id                                     100.000000
orig_projected_additional_accrued_interest     99.617331
hardship_dpd                                   99.517097
hardship_status                                99.517097
deferral_term                                  99.517097
hardship_amount                                99.517097
hardship_start_date                            99.517097
hardship_end_date                              99.517097
payment_plan_start_date                        99.517097
hardship_length                                99.517097
hardship_loan_status                           99.517097
hardship_type                                  99.517097
hardship_payoff_balance_amount                 99.517097
hardship_last_payment_amount                   99.517097
hardship_reason                                99.517097
debt_settlement_flag_date                      98.485160
settlement_status                              98.485160
settlement_date                